# 📊 Case Study: Análisis de Desempeño Comercial y Rentabilidad (2024–2025)
**Cliente/Empresa:** Andes Retail Group  
**Autor:** William  
**Herramientas:** Python (Pandas, NumPy), Power BI, Power Query (M), DAX  

---

## Contexto del Negocio y Objetivos

Andes Retail Group es una cadena minorista con presencia internacional en **Perú, Chile y Colombia**, comercializando productos en diversas categorías y segmentos de cliente (*Premium, Estándar y Económico*). 

Este proyecto busca evaluar el comportamiento comercial y financiero durante el periodo **2024–2025** para responder a preguntas estratégicas:
* ¿Cómo se distribuyen los ingresos y rentabilidades por país, categoría y segmento?
* ¿Qué patrones estacionales afectan el volumen de ventas?
* ¿Dónde se encuentran las mayores oportunidades de optimización y crecimiento comercial?

---

##  Canal de Preparación de Datos y Trazabilidad (ETL)

Para garantizar la integridad y auditabilidad de los datos entre el procesamiento de código en **Python** y el modelo multidimensional de **Power BI**, se aplicaron las siguientes reglas de negocio y transformaciones de datos:


### Reglas de Transformación:
* **Tipado estricto:** Normalización de fechas a tipo `datetime` y conversión explícita de montos monetarios a tipo numérico.
* **Nivel_Venta (Regla Condicional):** Clasificación basada en: `Si Ingresos >= 1000 -> "Venta Alta", sino -> "Venta Baja"`.
* **Métricas Financieras Calculadas:**
  * Ganancia = Ingresos - Costo
  * Margen (%) = Ganancia / Ingresos
* **Mapeo Estacional:** Creación de orden numérico para el análisis temporal (`Verano`: 1, `Otoño`: 2, `Invierno`: 3, `Primavera`: 4).

In [1]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. Carga de Dataset y Configuración de Tipos de Datos
# ---------------------------------------------------------
# Carga del dataset comercial
df = pd.read_csv('Andes_Retail_Group_2024_2025.csv') # Ajustar ruta de archivo según entorno

# Conversión explícita de tipos
df['Fecha_Pedido'] = pd.to_datetime(df['Fecha_Pedido'])

cols_financieras = ['Ingresos', 'Costo', 'Precio_Unitario', 'Unidades_Vendidas']
for col in cols_financieras:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# ---------------------------------------------------------
# 2. Creación de Columnas Calculadas y Reglas de Negocio
# ---------------------------------------------------------
# Regla condicional Nivel_Venta
df['Nivel_Venta'] = np.where(df['Ingresos'] >= 1000, 'Venta Alta', 'Venta Baja')

# Cálculo de Ganancia y Margen
df['Ganancia'] = df['Ingresos'] - df['Costo']
df['Margen (%)'] = df['Ganancia'] / df['Ingresos']

# Mapeo secuencial para análisis de estaciones
mapa_estaciones = {'Verano': 1, 'Otoño': 2, 'Invierno': 3, 'Primavera': 4}
df['Estación_Orden'] = df['Estación'].map(mapa_estaciones)

# ---------------------------------------------------------
# 3. Muestra de Trazabilidad y Auditoría de Datos
# ---------------------------------------------------------
print("=== EVIDENCIA 1: VALIDACIÓN DE TIPOS DE DATOS ===")
print(df[['Fecha_Pedido', 'Ingresos', 'Costo', 'Precio_Unitario']].dtypes)

print("\n=== EVIDENCIA 2: PRIMERAS FILAS DE TRANSFORMACIÓN ===")
cols_auditoria = ['Fecha_Pedido', 'Ingresos', 'Costo', 'Ganancia', 'Margen (%)', 'Nivel_Venta', 'Estación_Orden']
print(df[cols_auditoria].head())

=== EVIDENCIA 1: VALIDACIÓN DE TIPOS DE DATOS ===
Fecha_Pedido       datetime64[ns]
Ingresos                    int64
Costo                     float64
Precio_Unitario             int64
dtype: object

=== EVIDENCIA 2: PRIMERAS FILAS DE TRANSFORMACIÓN ===
  Fecha_Pedido  Ingresos    Costo  Ganancia  Margen (%) Nivel_Venta  \
0   2025-10-29       469   325.44    143.56    0.306098  Venta Baja   
1   2025-02-20      2156  1435.76    720.24    0.334063  Venta Alta   
2   2024-04-29      1148   710.19    437.81    0.381368  Venta Alta   
3   2024-09-21      1204   758.05    445.95    0.370390  Venta Alta   
4   2025-02-18      1395   890.08    504.92    0.361950  Venta Alta   

   Estación_Orden  
0               4  
1               1  
2               2  
3               4  
4               1  


---

## 3. Matriz de Validación de Métricas (Python vs. Power BI)

Para respaldar el modelo analítico construido en **Power BI Desktop**, se validan las agregaciones globales resultantes en Python:

In [2]:
# Validación de KPIs Principales
ingresos_totales = df['Ingresos'].sum()
ganancia_total = df['Ganancia'].sum()
margen_promedio = (ganancia_total / ingresos_totales) * 100
total_pedidos = df['ID_Pedido'].count()
clientes_unicos = df['ID_Cliente'].nunique()

print(f"📌 Ingresos Totales: ${ingresos_totales/1e6:.2f}M")
print(f"📌 Ganancia Total: ${ganancia_total/1e6:.2f}M")
print(f"📌 Margen de Ganancia Global: {margen_promedio:.2f}%")
print(f"📌 Total de Pedidos: {total_pedidos:,}")
print(f"📌 Clientes Únicos (Distinct Count): {clientes_unicos:,}")

📌 Ingresos Totales: $5.53M
📌 Ganancia Total: $1.94M
📌 Margen de Ganancia Global: 35.10%
📌 Total de Pedidos: 5,000
📌 Clientes Únicos (Distinct Count): 3,821


---

## 4. Arquitectura y Planificación del Dashboard en Power BI

El informe interactivo en Power BI se estructuró en dos niveles analíticos:

### 4.1 Vista 1: Overview Ejecutivo (Desempeño Macro)
* **Propósito:** Ofrecer una visión de alto nivel sobre la salud comercial global.
* **KPIs Principales (Tarjetas con 2 decimales):**
  * **Ingresos Totales:** `$5.53M`
  * **Ganancia Total:** `$1.94M`
  * **Unidades Vendidas:** Volúmenes operativos acumulados.
* **Visualizaciones:**
  * **Evolución Mensual:** Gráfico de líneas (2024-2025) para tendencias temporales.
  * **Distribución Geográfica:** Gráfico de barras de Ingresos por País (destacando a Perú como líder).
  * **Análisis por Segmento:** Gráfico de barras de Ingresos por Segmento de Cliente.

### 4.2 Vista 2: Análisis Detallado y Rentabilidad
* **Propósito:** Diagnosticar causas subyacentes, estacionalidad y comportamiento de segmentos.
* **Visualizaciones:**
  * **Análisis Estacional:** Gráfico combinado de barras (*Ingresos*) y líneas (*Margen %*) por Estación.
  * **Evolución de Rentabilidad:** Gráfico de líneas de Ganancia por Segmento de Cliente en el tiempo.
  * **Tabla Granular Auditáble:** Matriz detallada con formato condicional en Margen %, incorporando:
    * `País` | `Categoría_Producto` | `Recuento de ID_Pedido` | `Clientes Únicos (DISTINCTCOUNT)` | `Ingresos` | `Ganancia` | `Margen %`
* **Filtros e Interactividad:** Segmentadores por Fecha, País, Región, Categoría de Producto y Segmento de Cliente.

---

## 5. Caso de Negocio y Narrativa Ejecutiva (Modelo SCQA)

### 5.1 Vista General (Overview)
* **Situation (Situación):** Durante el periodo 2024–2025, Andes Retail Group alcanzó **\$5.53M en ingresos totales** y **\$1.94M** en ganancia**, con un margen general saludable y estable de **35.10%**.
* **Complication (Complicación):** Existe una marcada concentración comercial y estacional: **Perú y Chile generan cerca del 75% del ingreso total**, mientras que los segmentos *Premium* y *Estándar* capturan el **92%** de las ventas (dejando al segmento *Económico* con apenas el ~8%). Adicionalmente, se observan picos pronunciados en enero y caídas drásticas durante los meses de invierno.
* **Question (Pregunta):** ¿A qué se deben las caídas invernales y sobre qué variables se debe intervenir para estabilizar el flujo de caja sin comprometer la rentabilidad?
* **Answer (Respuesta):** La contracción de invierno es un problema puramente de **volumen de ventas y no de margen de ganancia** (el margen se mantiene rígido entre 34.85% y 35.30% a lo largo de todo el año). Dado que Verano genera **3.43 veces los ingresos de Invierno** (Invierno está un **~71% por debajo** de Verano), existe una oportunidad clave para lanzar campañas de incentivo comercial en invierno enfocadas en los segmentos *Premium* y *Estándar*.

### 5.2 Vista Detallada
* **Situation (Situación):** La eficiencia financiera es consistente en todas las combinaciones geográficas y de producto (márgenes individuales entre 34.79% y 35.38%).
* **Complication (Complicación):** El comportamiento del segmento *Económico* permanece plano durante todo el año, mientras que la variable de Región (*Norte, Sur, Centro*) no genera variación explicativa sobre las caídas estacionales.
* **Question (Pregunta):** ¿Dónde se debe focalizar la fuerza de ventas para maximizar el retorno de inversión en temporadas bajas?
* **Answer (Respuesta):** Se debe priorizar la categoría de **Electrónica en Perú** (que alcanza el margen más alto del portafolio con un **35.38%**) apoyada por activaciones comerciales sobre la base de clientes *Premium*.

---

## 6. Comunicación Asincrónica para Stakeholders (Slack/Email)

> **Subject / Headline:** 📊 Resumen Ejecutivo: Desempeño Comercial 2024–2025 & Plan de Acción
> 
> Hola a todos, compartimos los hallazgos principales del cierre del análisis comercial:
>
> * 📈 **Ingresos & Rentabilidad:** Cerramos con **\$5.53M en ingresos totales** y **\$1.94M** en ganancia** (Margen global: **35.10%**).
> * 🌍 **Concentración:** Perú y Chile representan el **~75%** de los ingresos.
> * ❄️ **Oportunidad Clave (Estacionalidad):** El volumen en Verano supera en **3.43×** a Invierno (Invierno se encuentra **71% por debajo**). Al mantenerse el margen estable en 35% en ambas estaciones, la caída invernal es una oportunidad directa de expansión de volumen.
> * 🎯 **Target Estratégico:** Los segmentos *Premium* y *Estándar* explican el **92%** de las ventas.
>
> 💡 **Recomendación:** Lanzar una campaña de aceleración de volumen durante la temporada de Invierno orientada a clientes *Premium* y *Estándar*, priorizando la categoría de **Electrónica en Perú** (máxima rentabilidad con **35.38%**).
> 